In [ ]:
import netCDF4

In [ ]:
import glob
import pandas as pd
fileall = glob.glob("../disdrodb-data/DISDRODB/Processed/EPFL/EPFL_2009/L0B/??/L0B.EPFL_2009.??.*.V0.nc")
for file in fileall:
    nc = netCDF4.Dataset(file,"r")
    print(file, nc["raw_drop_number"][:].mean())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
'''
for campaign in ["EPFL_ROOF_2008", "EPFL_2009", "EPFL_ROOF_2010",
                 "EPFL_ROOF_2011", "EPFL_ROOF_2012", "COMMON_2011"]:

    fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
    for i, infile in enumerate(fileall):
        #infile = "../disdrodb-data/DISDRODB/Processed/EPFL/EPFL_2009/L0B/31/L0B.EPFL_2009.31.s20090325000000.e20090325003340.V0.nc"
        nc = netCDF4.Dataset(infile,"r")
        plt.figure()
        plt.imshow(np.sum(nc["raw_drop_number"][:], axis=2))
        plt.savefig(f"rawdata_velocity_marginalized_{campaign}_data{i}")
        plt.close()
        plt.figure()
        plt.bar(x=nc["diameter_bin_center"][:], height=np.sum(np.sum(nc["raw_drop_number"][:], axis=2), axis=0)/nc["diameter_bin_width"][:], width=0.1)
        maximum = np.fmax(50, np.max(np.sum(np.sum(nc["raw_drop_number"][:], axis=2), axis=0)/nc["diameter_bin_width"][:]))
        plt.xlim(0, 3)
        plt.ylim(0, maximum*1.05)
        plt.savefig(f"histogram_{campaign}_data{i}")
        plt.close()
'''

In [ ]:
campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile,"r")
plt.figure()
plt.imshow(np.sum(nc["raw_drop_number"][:], axis=2))
plt.savefig(f"rawdata_velocity_marginalized_{campaign}_detail")
plt.close()
plt.figure()
plt.bar(x=nc["diameter_bin_center"][:], height=np.sum(np.sum(nc["raw_drop_number"][:], axis=2), axis=0)/nc["diameter_bin_width"][:], width=0.1)
maximum = np.fmax(50, np.max(np.sum(np.sum(nc["raw_drop_number"][:], axis=2), axis=0)/nc["diameter_bin_width"][:]))
plt.xlim(0, 3)
plt.ylim(0, maximum*1.05)
plt.title("20111010 00:00:00-00:49:30")
plt.savefig(f"histogram_{campaign}_detail")
plt.title("")
plt.close()

In [ ]:
for i in range(100):
    start = i*30
    end = (i+1)*30
    start_min = start // 60
    start_sec = start % 60
    end_min = end // 60
    end_sec = end % 60
    plt.figure()
    plt.bar(x=nc["diameter_bin_center"][:], height=np.sum(nc["raw_drop_number"][:], axis=2)[i]/nc["diameter_bin_width"][:], width=0.1)
    #maximum = np.fmax(5, np.max(np.sum(nc["raw_drop_number"][:], axis=2)[i]/nc["diameter_bin_width"][:]))
    maximum = 450
    plt.ylabel("# of drops / binsize[mm]")
    plt.xlim(0, 3)
    plt.ylim(0, maximum*1.05)
    plt.title(f"20111010 00:{start_min:02d}:{start_sec:02d}-00:{end_min:02d}:{end_sec:02d}")
    plt.savefig(f"histogram_{campaign}_data2_time{i:02d}_detail")

In [ ]:
# case 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gamma
from scipy.optimize import minimize
from scipy.optimize import minimize_scalar

# Example histogram data
# Unequal bin sizes: first 10 bins have width 0.125, next 10 bins have width 0.25, etc.
'''
bin_edges = np.concatenate([
    np.linspace(0, 1.25, 11),  # 10 bins of width 0.125
    np.linspace(1.5, 4, 11)[1:],  # 10 bins of width 0.25
    np.linspace(4.5, 8, 12)[1:]  # 12 bins of width 0.5
])
'''

D1list = []
D2list = []
D3list = []
fitted_shape1_list = []
fitted_scale1_list = []

c0 = -0.0429
c1 = 0.0307
c2 = 0.2946

for i in range(100):
    print(f"{i=}")
    # Synthetic histogram values (you should replace this with your actual histogram data)
    hist_values = np.sum(nc["raw_drop_number"][:], axis=2)[i][:20]
    #hist_values = np.random.poisson(10, size=len(bin_edges) - 1)

    # Compute bin centers and bin widths
    bin_centers = nc["diameter_bin_center"][:][:20]
    bin_widths = nc["diameter_bin_width"][:][:20]
    #bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    #bin_widths = np.diff(bin_edges)

    # Normalize histogram values
    hist_values_normalized = hist_values / (hist_values.sum() * bin_widths)
    
    def negative_log_likelihood_binned(params, bin_edges, hist_values):
        shape, scale = params
        if shape <= 0 or scale <= 1e-6:
            return np.inf
        # Compute cumulative probabilities for bin edges
        cdf_values = gamma.cdf(bin_edges, a=shape, scale=scale)
        prob_masses = np.diff(cdf_values)  # Probability mass for each bin
        #prob_masses = np.maximum(prob_masses, 1e-10)  # Avoid log(0)
        return -np.sum(hist_values * np.log(prob_masses))

    # Define the negative log-likelihood function for fitting the Gamma distribution
    def negative_log_likelihood(params, x, y, bin_widths):
        shape, scale = params
        if shape <= 0 or scale <= 1e-6:
            return np.inf
        pdf_values = gamma.pdf(x, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(y * np.log(pdf_values) - pdf_values)
    
    def negative_log_likelihood_constrained_scalar(shape):
        #if shape <= 0:
        #    return np.inf
        mu = shape - 1
        scale = 1/(1.935 + 0.735 * mu + 0.0365 * mu**2)
        if scale <= 1e-6:  # Avoid scale being too small
            return np.inf
        pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(hist_values * np.log(pdf_values) - pdf_values)
    
    def negative_log_likelihood_constrained_scalar2(shape):
        #if shape <= 0:
        #    return np.inf
        mu = shape - 1
        scale = 1/(c0 + c1 * mu + c2 * mu**2)
        if scale <= 1e-6:  # Avoid scale being too small
            return np.inf
        pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(hist_values * np.log(pdf_values) - pdf_values)

    # Initial guess for shape and scale parameters
    initial_guess1 = [3.0, 3.0]
    initial_guess2 = [1.0]
    initial_guess3 = [6.0]

    # Fit the Gamma distribution
    result1 = minimize(
        negative_log_likelihood,
        initial_guess1,
        args=(bin_centers, hist_values, bin_widths),
        method='BFGS',
        options={"gtol":1e-1},
        #bounds=[(1e-6, None), (1e-6, None)]
    )
    result2 = minimize_scalar(negative_log_likelihood_constrained_scalar)
    result3 = minimize_scalar(negative_log_likelihood_constrained_scalar2)

    if result1.success:
        fitted_shape1, fitted_scale1 = result1.x
        print(f"{result1=}")
        mu1 = fitted_shape1-1
        D1 = (mu1+3.67)*fitted_scale1
        D1list.append(D1)
        print(f"Fitted Gamma parameters: shape={fitted_shape1}, scale={fitted_scale1}")
        fitted_scale1_list.append(fitted_scale1)
        fitted_shape1_list.append(fitted_shape1)
    else:
        raise RuntimeError("Optimization failed")
        
    if result2.success:
        fitted_shape2 = result2.x
        print(f"{result2=}")
        mu2 = fitted_shape2 - 1
        fitted_scale2 = 1/(1.935 + 0.735 * mu2 + 0.0365 * mu2**2)
        D2 = (mu2+3.67)*fitted_scale2
        D2list.append(D2)
        print(f"Fitted Gamma parameters: shape={fitted_shape2}, scale={fitted_scale2}")
    else:
        raise RuntimeError("Optimization failed")
        
    if result3.success:
        fitted_shape3 = result3.x
        print(f"{result3=}")
        mu3 = fitted_shape3 - 1
        fitted_scale3 = 1/(c0 + c1 * mu3 + c2 * mu3**2)
        D3 = (mu3+3.67)*fitted_scale3
        D3list.append(D3)
        print(f"Fitted Gamma parameters: shape={fitted_shape3}, scale={fitted_scale3}")
    else:
        raise RuntimeError("Optimization failed")

    # Generate the fitted Gamma PDF for plotting
    #x_values = np.linspace(bin_edges[0], bin_edges[-1], 1000)
    x_values = np.linspace(0, 5, 501)
    fitted_pdf1 = gamma.pdf(x_values, a=fitted_shape1, scale=fitted_scale1)
    fitted_pdf2 = gamma.pdf(x_values, a=fitted_shape2, scale=fitted_scale2)
    fitted_pdf3 = gamma.pdf(x_values, a=fitted_shape3, scale=fitted_scale3)

    # Plot the histogram and the fitted Gamma PDF
    plt.figure(figsize=(8, 5))
    plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")
    plt.plot(x_values, fitted_pdf1, label=f"Fitted Gamma PDF\n(shape={fitted_shape1:.2f}, scale={fitted_scale1:.2f})", color="red")
    plt.plot(x_values, fitted_pdf2, label=f"Fitted Gamma PDF\n(shape={fitted_shape2:.2f}, scale={fitted_scale2:.2f})", color="green")
    plt.plot(x_values, fitted_pdf3, label=f"Fitted Gamma PDF\n(shape={fitted_shape3:.2f}, scale={fitted_scale3:.2f})", color="blue")
    plt.annotate(f"{D1=:.2f} mm", [0.7, 0.55], xycoords='axes fraction', color="red")
    plt.annotate(f"{D2=:.2f} mm", [0.7, 0.475], xycoords='axes fraction', color="green")
    plt.annotate(f"{D3=:.2f} mm", [0.7, 0.4], xycoords='axes fraction', color="blue")
    plt.xlim(0, 5)
    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.legend()
    plt.title("EPFL, COMMON 2011")
    #plt.show()
    plt.savefig(f"GammaFit_case1_step{i}")

In [ ]:
X = [0, 3]
plt.figure(figsize=(6, 6))
plt.scatter(D1list, D2list, color="g")
plt.scatter(D1list, D3list, color="b")
plt.xlim(0, 2.5)
plt.ylim(0, 2.5)
plt.plot(X, X)
plt.xlabel("D0 with full Gamma")
plt.ylabel("D0 with constrained Gamma")
plt.savefig("D0_comparison_case1")

In [ ]:
from scipy.optimize import curve_fit
def quadratic_model(x, a, b, c):
    return a * x**2 + b * x + c

params, covariance = curve_fit(quadratic_model, np.array(fitted_shape1_list), 1/np.array(fitted_scale1_list))
D0 = (np.array(fitted_shape1_list)-1+3.67)*np.array(fitted_scale1_list)

fitted_shape = np.linspace(0, 20, 201)
mu = fitted_shape - 1
fitted_scale = 1/(1.935 + 0.735 * mu + 0.0365 * mu**2)
D02 = (mu+3.67)*np.array(fitted_scale)

fitted_shape2 = np.linspace(0, 20, 201)
fitted_scale2 = 1/(c0+c1*mu+c2*mu*mu)
D03 = (mu+3.67)*np.array(fitted_scale2)

plt.figure(figsize=(9, 6))
mappable = plt.scatter(np.array(fitted_shape1_list)-1, 1/np.array(fitted_scale1_list),
                       c=D0, label="EPFL COMMON_2011", vmin=1.0, vmax=1.6)
cbar = plt.colorbar(mappable)
cbar.set_label("D0[mm]", fontsize=14)
cbar.set_ticklabels([1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6], fontsize=14)
plt.plot(fitted_shape, 1/fitted_scale, label=r"$\Lambda=1.935+0.735\mu+0.0365\mu^2$")
plt.scatter(fitted_shape, 1/fitted_scale, c=D02,
            vmin=1.0, vmax=1.6)

plt.plot(fitted_shape2, 1/fitted_scale2, label=r"new fit")
plt.scatter(fitted_shape2, 1/fitted_scale2, c=D03,
            vmin=1.0, vmax=1.6)

plt.ylim(0, 20)
plt.xlabel(r"$\mu$", fontsize=14)
plt.ylabel(r"$\Lambda$", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
plt.savefig("mu_Lambda_relation_case1")

In [ ]:
# case2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gamma
from scipy.optimize import minimize
from scipy.optimize import minimize_scalar

# Example histogram data
# Unequal bin sizes: first 10 bins have width 0.125, next 10 bins have width 0.25, etc.
'''
bin_edges = np.concatenate([
    np.linspace(0, 1.25, 11),  # 10 bins of width 0.125
    np.linspace(1.5, 4, 11)[1:],  # 10 bins of width 0.25
    np.linspace(4.5, 8, 12)[1:]  # 12 bins of width 0.5
])
'''

D1list = []
D2list = []
D3list = []
fitted_shape1_list = []
fitted_scale1_list = []

funlist1 = []
funlist2 = []
funlist3 = []

c0 = -2.6744
c1 = 1.2012
c2 = 0.0318

for i in range(100):
    print(f"{i=}")
    # Synthetic histogram values (you should replace this with your actual histogram data)
    hist_values = np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20]
    #hist_values = np.random.poisson(10, size=len(bin_edges) - 1)

    # Compute bin centers and bin widths
    bin_centers = nc["diameter_bin_center"][:][4:20]
    bin_widths = nc["diameter_bin_width"][:][4:20]
    #bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    #bin_widths = np.diff(bin_edges)

    # Normalize histogram values
    hist_values_normalized = hist_values / (hist_values.sum() * bin_widths)
    
    def negative_log_likelihood_binned(params, bin_edges, hist_values):
        shape, scale = params
        if shape <= 0 or scale <= 1e-6:
            return np.inf
        # Compute cumulative probabilities for bin edges
        cdf_values = gamma.cdf(bin_edges, a=shape, scale=scale)
        prob_masses = np.diff(cdf_values)  # Probability mass for each bin
        #prob_masses = np.maximum(prob_masses, 1e-10)  # Avoid log(0)
        return -np.sum(hist_values * np.log(prob_masses))

    # Define the negative log-likelihood function for fitting the Gamma distribution
    def negative_log_likelihood(params, x, y, bin_widths):
        shape, scale = params
        if shape <= 0 or scale <= 1e-6:
            return np.inf
        pdf_values = gamma.pdf(x, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(y * np.log(pdf_values) - pdf_values)
    
    def negative_log_likelihood_constrained_scalar(shape):
        #if shape <= 0:
        #    return np.inf
        mu = shape - 1
        scale = 1/(1.935 + 0.735 * mu + 0.0365 * mu**2)
        if scale <= 1e-6:  # Avoid scale being too small
            return np.inf
        pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(hist_values * np.log(pdf_values) - pdf_values)
    
    def negative_log_likelihood_constrained_scalar2(shape):
        #if shape <= 0:
        #    return np.inf
        mu = shape - 1
        scale = 1/(c0 + c1 * mu + c2 * mu**2)
        if scale <= 1e-6:  # Avoid scale being too small
            return np.fmax(1/scale, 1e6)
        pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(hist_values * np.log(pdf_values) - pdf_values)

    # Initial guess for shape and scale parameters
    initial_guess1 = [3.0, 3.0]
    initial_guess2 = [1.0]
    initial_guess3 = [9.0]

    # Fit the Gamma distribution
    result1 = minimize(
        negative_log_likelihood,
        initial_guess1,
        args=(bin_centers, hist_values, bin_widths),
        method='BFGS',
        options={"gtol":1e-1},
        #bounds=[(1e-6, None), (1e-6, None)]
    )
    result2 = minimize_scalar(negative_log_likelihood_constrained_scalar)
    result3 = minimize_scalar(negative_log_likelihood_constrained_scalar2, bounds=(3, 100), method='bounded')

    if result1.success:
        fitted_shape1, fitted_scale1 = result1.x
        print(f"{result1=}")
        mu1 = fitted_shape1-1
        D1 = (mu1+3.67)*fitted_scale1
        D1list.append(D1)
        print(f"Fitted Gamma parameters: shape={fitted_shape1}, scale={fitted_scale1}")
        fitted_scale1_list.append(fitted_scale1)
        fitted_shape1_list.append(fitted_shape1)
        funlist1.append(result1.fun)
    else:
        raise RuntimeError("Optimization failed")
        
    if result2.success:
        fitted_shape2 = result2.x
        print(f"{result2=}")
        mu2 = fitted_shape2 - 1
        fitted_scale2 = 1/(1.935 + 0.735 * mu2 + 0.0365 * mu2**2)
        D2 = (mu2+3.67)*fitted_scale2
        D2list.append(D2)
        print(f"Fitted Gamma parameters: shape={fitted_shape2}, scale={fitted_scale2}")
        funlist2.append(result2.fun)
    else:
        raise RuntimeError("Optimization failed")
        
    if result3.success:
        fitted_shape3 = result3.x
        print(f"{result3=}")
        mu3 = fitted_shape3 - 1
        fitted_scale3 = 1/(c0 + c1 * mu3 + c2 * mu3**2)
        D3 = (mu3+3.67)*fitted_scale3
        D3list.append(D3)
        print(f"Fitted Gamma parameters: shape={fitted_shape3}, scale={fitted_scale3}")
        funlist3.append(result3.fun)
    else:
        raise RuntimeError("Optimization failed")

    # Generate the fitted Gamma PDF for plotting
    #x_values = np.linspace(bin_edges[0], bin_edges[-1], 1000)
    x_values = np.linspace(0, 5, 501)
    fitted_pdf1 = gamma.pdf(x_values, a=fitted_shape1, scale=fitted_scale1)
    fitted_pdf2 = gamma.pdf(x_values, a=fitted_shape2, scale=fitted_scale2)
    fitted_pdf3 = gamma.pdf(x_values, a=fitted_shape3, scale=fitted_scale3)

    # Plot the histogram and the fitted Gamma PDF
    plt.figure(figsize=(8, 5))
    plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")
    plt.plot(x_values, fitted_pdf1, label=f"Fitted Gamma PDF\n(shape={fitted_shape1:.2f}, scale={fitted_scale1:.2f})", color="red")
    plt.plot(x_values, fitted_pdf2, label=f"Fitted Gamma PDF\n(shape={fitted_shape2:.2f}, scale={fitted_scale2:.2f})", color="green")
    plt.plot(x_values, fitted_pdf3, label=f"Fitted Gamma PDF\n(shape={fitted_shape3:.2f}, scale={fitted_scale3:.2f})", color="blue")
    plt.annotate(f"{D1=:.2f} mm", [0.7, 0.55], xycoords='axes fraction', color="red")
    plt.annotate(f"{D2=:.2f} mm", [0.7, 0.475], xycoords='axes fraction', color="green")
    plt.annotate(f"{D3=:.2f} mm", [0.7, 0.4], xycoords='axes fraction', color="blue")
    plt.xlim(0, 5)
    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.legend()
    plt.title("EPFL, COMMON 2011")
    #plt.show()
    plt.savefig(f"GammaFit_case2_step{i}")

In [ ]:
X = [0, 3]
plt.figure(figsize=(6, 6))
plt.scatter(D1list, D2list, color="g", zorder=1)
plt.scatter(D1list, D3list, color="b", zorder=0)
plt.xlim(0, 2.5)
plt.ylim(0, 2.5)
plt.plot(X, X)
plt.xlabel("D0 with full Gamma")
plt.ylabel("D0 with constrained Gamma")
plt.savefig("D0_comparison_case2")

print(f"{np.sum(funlist1)=}")
print(f"{np.sum(funlist2)=}")
print(f"{np.sum(funlist3)=}")

In [ ]:
from scipy.optimize import curve_fit
def quadratic_model(x, a, b, c):
    return a * x**2 + b * x + c

params, covariance = curve_fit(quadratic_model, np.array(fitted_shape1_list), 1/np.array(fitted_scale1_list))
D0 = (np.array(fitted_shape1_list)-1+3.67)*np.array(fitted_scale1_list)

fitted_shape = np.linspace(0, 20, 201)
mu = fitted_shape - 1
fitted_scale = 1/(1.935 + 0.735 * mu + 0.0365 * mu**2)
D02 = (mu+3.67)*np.array(fitted_scale)

fitted_shape2 = np.linspace(0, 20, 201)
fitted_scale2 = 1/(c0+c1*mu+c2*mu*mu)
D03 = (mu+3.67)*np.array(fitted_scale2)

plt.figure(figsize=(9, 6))
mappable = plt.scatter(np.array(fitted_shape1_list)-1, 1/np.array(fitted_scale1_list),
                       c=D0, label="EPFL COMMON_2011", vmin=1.0, vmax=2.5)
cbar = plt.colorbar(mappable)
cbar.set_label("D0[mm]", fontsize=14)
cbar.set_ticklabels([1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4], fontsize=14)
plt.plot(fitted_shape, 1/fitted_scale, label=r"$\Lambda=1.935+0.735\mu+0.0365\mu^2$")
plt.scatter(fitted_shape, 1/fitted_scale, c=D02,
            vmin=1.0, vmax=2.5)

plt.plot(fitted_shape2, 1/fitted_scale2, label=r"new fit")
plt.scatter(fitted_shape2, 1/fitted_scale2, c=D03,
            vmin=1.0, vmax=2.5)

plt.ylim(0, 20)
plt.xlabel(r"$\mu$", fontsize=14)
plt.ylabel(r"$\Lambda$", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
plt.savefig("mu_Lambda_relation_case2")

# test data

In [ ]:
campaign = "DAVOS_2009_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[1]
nc = netCDF4.Dataset(infile,"r")

In [ ]:
D1list = []
D2list = []
D3list = []
fitted_shape1_list = []
fitted_scale1_list = []

funlist1 = []
funlist2 = []
funlist3 = []

c0 = -2.6744
c1 = 1.2012
c2 = 0.0318

for i in range(100):
    print(f"{i=}")
    # Synthetic histogram values (you should replace this with your actual histogram data)
    hist_values = np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20]
    #hist_values = np.random.poisson(10, size=len(bin_edges) - 1)

    # Compute bin centers and bin widths
    bin_centers = nc["diameter_bin_center"][:][4:20]
    bin_widths = nc["diameter_bin_width"][:][4:20]
    #bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    #bin_widths = np.diff(bin_edges)

    # Normalize histogram values
    hist_values_normalized = hist_values / (hist_values.sum() * bin_widths)
    
    def negative_log_likelihood_binned(params, bin_edges, hist_values):
        shape, scale = params
        if shape <= 0 or scale <= 1e-6:
            return np.inf
        # Compute cumulative probabilities for bin edges
        cdf_values = gamma.cdf(bin_edges, a=shape, scale=scale)
        prob_masses = np.diff(cdf_values)  # Probability mass for each bin
        #prob_masses = np.maximum(prob_masses, 1e-10)  # Avoid log(0)
        return -np.sum(hist_values * np.log(prob_masses))

    # Define the negative log-likelihood function for fitting the Gamma distribution
    def negative_log_likelihood(params, x, y, bin_widths):
        shape, scale = params
        if shape <= 0 or scale <= 1e-6:
            return np.inf
        pdf_values = gamma.pdf(x, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(y * np.log(pdf_values) - pdf_values)
    
    def negative_log_likelihood_constrained_scalar(shape):
        #if shape <= 0:
        #    return np.inf
        mu = shape - 1
        scale = 1/(1.935 + 0.735 * mu + 0.0365 * mu**2)
        if scale <= 1e-6:  # Avoid scale being too small
            return np.inf
        pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(hist_values * np.log(pdf_values) - pdf_values)
    
    def negative_log_likelihood_constrained_scalar2(shape):
        #if shape <= 0:
        #    return np.inf
        mu = shape - 1
        scale = 1/(c0 + c1 * mu + c2 * mu**2)
        if scale <= 1e-6:  # Avoid scale being too small
            return np.fmax(1/scale, 1e6)
        pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
        #pdf_values = np.maximum(pdf_values, 1e-100)
        return -np.sum(hist_values * np.log(pdf_values) - pdf_values)

    # Initial guess for shape and scale parameters
    initial_guess1 = [3.0, 3.0]
    initial_guess2 = [1.0]
    initial_guess3 = [9.0]

    # Fit the Gamma distribution
    result1 = minimize(
        negative_log_likelihood,
        initial_guess1,
        args=(bin_centers, hist_values, bin_widths),
        method='BFGS',
        options={"gtol":1e-1},
        #bounds=[(1e-6, None), (1e-6, None)]
    )
    result2 = minimize_scalar(negative_log_likelihood_constrained_scalar)
    result3 = minimize_scalar(negative_log_likelihood_constrained_scalar2, bounds=(3, 100), method='bounded')

    if result1.success:
        fitted_shape1, fitted_scale1 = result1.x
        print(f"{result1=}")
        mu1 = fitted_shape1-1
        D1 = (mu1+3.67)*fitted_scale1
        D1list.append(D1)
        print(f"Fitted Gamma parameters: shape={fitted_shape1}, scale={fitted_scale1}")
        fitted_scale1_list.append(fitted_scale1)
        fitted_shape1_list.append(fitted_shape1)
        funlist1.append(result1.fun)
    else:
        raise RuntimeError("Optimization failed")
        
    if result2.success:
        fitted_shape2 = result2.x
        print(f"{result2=}")
        mu2 = fitted_shape2 - 1
        fitted_scale2 = 1/(1.935 + 0.735 * mu2 + 0.0365 * mu2**2)
        D2 = (mu2+3.67)*fitted_scale2
        D2list.append(D2)
        print(f"Fitted Gamma parameters: shape={fitted_shape2}, scale={fitted_scale2}")
        funlist2.append(result2.fun)
    else:
        raise RuntimeError("Optimization failed")
        
    if result3.success:
        fitted_shape3 = result3.x
        print(f"{result3=}")
        mu3 = fitted_shape3 - 1
        fitted_scale3 = 1/(c0 + c1 * mu3 + c2 * mu3**2)
        D3 = (mu3+3.67)*fitted_scale3
        D3list.append(D3)
        print(f"Fitted Gamma parameters: shape={fitted_shape3}, scale={fitted_scale3}")
        funlist3.append(result3.fun)
    else:
        raise RuntimeError("Optimization failed")

    # Generate the fitted Gamma PDF for plotting
    #x_values = np.linspace(bin_edges[0], bin_edges[-1], 1000)
    x_values = np.linspace(0, 5, 501)
    fitted_pdf1 = gamma.pdf(x_values, a=fitted_shape1, scale=fitted_scale1)
    fitted_pdf2 = gamma.pdf(x_values, a=fitted_shape2, scale=fitted_scale2)
    fitted_pdf3 = gamma.pdf(x_values, a=fitted_shape3, scale=fitted_scale3)

    # Plot the histogram and the fitted Gamma PDF
    plt.figure(figsize=(8, 5))
    plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")
    plt.plot(x_values, fitted_pdf1, label=f"Fitted Gamma PDF\n(shape={fitted_shape1:.2f}, scale={fitted_scale1:.2f})", color="red")
    plt.plot(x_values, fitted_pdf2, label=f"Fitted Gamma PDF\n(shape={fitted_shape2:.2f}, scale={fitted_scale2:.2f})", color="green")
    plt.plot(x_values, fitted_pdf3, label=f"Fitted Gamma PDF\n(shape={fitted_shape3:.2f}, scale={fitted_scale3:.2f})", color="blue")
    plt.annotate(f"{D1=:.2f} mm", [0.7, 0.55], xycoords='axes fraction', color="red")
    plt.annotate(f"{D2=:.2f} mm", [0.7, 0.475], xycoords='axes fraction', color="green")
    plt.annotate(f"{D3=:.2f} mm", [0.7, 0.4], xycoords='axes fraction', color="blue")
    plt.xlim(0, 5)
    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.legend()
    plt.title("EPFL, DAVOS_2009_2011")
    #plt.show()
    plt.savefig(f"GammaFit_case2_testdata_step{i}")

In [ ]:
print(f"{np.sum(funlist1)=}")
print(f"{np.sum(funlist2)=}")
print(f"{np.sum(funlist3)=}")

In [ ]:
def quadratic_model(x, a, b, c):
    return a * x**2 + b * x + c

params, covariance = curve_fit(quadratic_model, np.array(fitted_shape1_list), 1/np.array(fitted_scale1_list))
D0 = (np.array(fitted_shape1_list)-1+3.67)*np.array(fitted_scale1_list)

fitted_shape = np.linspace(0, 20, 201)
mu = fitted_shape - 1
fitted_scale = 1/(1.935 + 0.735 * mu + 0.0365 * mu**2)
D02 = (mu+3.67)*np.array(fitted_scale)

fitted_shape2 = np.linspace(0, 20, 201)
fitted_scale2 = 1/(c0+c1*mu+c2*mu*mu)
D03 = (mu+3.67)*np.array(fitted_scale2)

plt.figure(figsize=(9, 6))
mappable = plt.scatter(np.array(fitted_shape1_list)-1, 1/np.array(fitted_scale1_list),
                       c=D0, label="DAVOS_2009_2011", vmin=1.0, vmax=2.5)
cbar = plt.colorbar(mappable)
cbar.set_label("D0[mm]", fontsize=14)
cbar.set_ticklabels([1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4], fontsize=14)
plt.plot(fitted_shape, 1/fitted_scale, label=r"$\Lambda=1.935+0.735\mu+0.0365\mu^2$")
plt.scatter(fitted_shape, 1/fitted_scale, c=D02,
            vmin=1.0, vmax=2.5)

plt.plot(fitted_shape2, 1/fitted_scale2, label=r"new fit")
plt.scatter(fitted_shape2, 1/fitted_scale2, c=D03,
            vmin=1.0, vmax=2.5)

plt.ylim(0, 20)
plt.xlabel(r"$\mu$", fontsize=14)
plt.ylabel(r"$\Lambda$", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
plt.savefig("mu_Lambda_relation_case2_testdata")

In [ ]:
X = [0, 3]
plt.figure(figsize=(6, 6))
plt.scatter(D1list, D2list, color="g", zorder=1)
plt.scatter(D1list, D3list, color="b", zorder=0)
plt.xlim(0, 2.5)
plt.ylim(0, 2.5)
plt.plot(X, X)
plt.xlabel("D0 with full Gamma")
plt.ylabel("D0 with constrained Gamma")
plt.savefig("D0_comparison_case2_testdata")

In [ ]:
plt.figure(figsize=(9, 6))
mappable = plt.scatter(np.array(fitted_shape1_list)-1, 1/np.array(fitted_scale1_list),
                       c=D0, label="EPFL COMMON_2011", vmin=1.0, vmax=1.6)
cbar = plt.colorbar(mappable)
cbar.set_label("D0[mm]", fontsize=14)
cbar.set_ticklabels([1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6], fontsize=14)
plt.plot(fitted_shape, 1/fitted_scale, label=r"$\Lambda=1.935+0.735\mu+0.0365\mu^2$")
plt.scatter(fitted_shape, 1/fitted_scale, c=D02,
            vmin=1.0, vmax=1.6)
plt.xlabel(r"$\mu$", fontsize=14)
plt.ylabel(r"$\Lambda$", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=14)
#plt.savefig("mu_Lambda_relation")

In [ ]:
plt.scatter(np.array(fitted_shape1_list)-1, 1/np.array(fitted_scale1_list),
            c=D0, label="EPFL COMMON_2011", vmin=1.0, vmax=1.6)

In [ ]:
import torch
#from scipy.stats import gamma
from torch.distributions.gamma import gamma

# Define the scale computation
def compute_scale(mu, c0, c1, c2):
    return c0 + c1 * mu + c2 * mu**2

# Compute the negative log-likelihood for a single histogram
def compute_negative_log_likelihood(mu, bin_centers, bin_widths, hist_values, c0, c1, c2):
    scale = compute_scale(mu, c0, c1, c2)
    shape = mu + 1
    if scale <= 1e-6:
        return torch.tensor(float('inf'))  # Avoid invalid scale
    pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
    pdf_values = torch.tensor(pdf_values, dtype=torch.float32)
    hist_values = torch.tensor(hist_values, dtype=torch.float32)
    return -torch.sum(hist_values * torch.log(pdf_values) - pdf_values)

# Optimize mu for a single dataset using manual gradients
def optimize_mu_manual(bin_centers, bin_widths, hist_values, c0, c1, c2, lr=1e-2, max_iter=100, tol=1e-6):
    # Initialize mu
    mu = torch.tensor(2.0, dtype=torch.float32, requires_grad=True)  # Initial guess

    for iteration in range(max_iter):
        # Compute the negative log-likelihood
        nll = compute_negative_log_likelihood(mu, bin_centers, bin_widths, hist_values, c0, c1, c2)

        # Compute gradients manually
        nll.backward()  # Compute the gradient with respect to mu

        # Update mu using gradient descent
        with torch.no_grad():
            mu -= lr * mu.grad

        # Clear the gradient for the next step
        mu.grad.zero_()

        # Check for convergence
        if abs(mu.grad.item()) < tol:
            break

    return mu.item()

# Compute the total negative log-likelihood for all datasets
def compute_total_log_likelihood(datasets, c0, c1, c2, lr_mu=1e-2, max_iter_mu=100):
    total_ll = 0.0
    for bin_centers, bin_widths, hist_values in datasets:
        # Optimize mu for the current histogram
        mu = optimize_mu_manual(bin_centers, bin_widths, hist_values, c0, c1, c2, lr=lr_mu, max_iter=max_iter_mu)
        scale = compute_scale(mu, c0, c1, c2)
        shape = mu + 1

        # Compute PDF values
        pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
        pdf_values = torch.tensor(pdf_values, dtype=torch.float32)
        hist_values = torch.tensor(hist_values, dtype=torch.float32)

        # Accumulate negative log-likelihood
        total_ll += -torch.sum(hist_values * torch.log(pdf_values) - pdf_values).item()

    return total_ll

# Gradient descent for c0, c1, c2 using PyTorch
def optimize_meta_params(datasets, initial_params, lr=1e-3, max_iter=100, tol=1e-6):
    # Initialize parameters as PyTorch tensors with gradients enabled
    c0 = torch.tensor(initial_params[0], dtype=torch.float32, requires_grad=True)
    c1 = torch.tensor(initial_params[1], dtype=torch.float32, requires_grad=True)
    c2 = torch.tensor(initial_params[2], dtype=torch.float32, requires_grad=True)

    optimizer = torch.optim.Adam([c0, c1, c2], lr=lr)

    for iteration in range(max_iter):
        # Zero the gradients
        optimizer.zero_grad()

        # Compute the total negative log-likelihood
        total_ll = compute_total_log_likelihood(datasets, c0, c1, c2)

        # Backpropagate to compute gradients
        total_ll.backward()

        # Update parameters
        optimizer.step()

        # Print progress
        print(f"Iteration {iteration}: c0={c0.item():.4f}, c1={c1.item():.4f}, c2={c2.item():.4f}, Total LL={total_ll:.4f}")

        # Check for convergence
        if abs(total_ll) < tol:
            print("Convergence achieved!")
            break

    return c0.item(), c1.item(), c2.item()

# Example usage
if __name__ == "__main__":
    # Replace this with actual histogram datasets: [(bin_centers, bin_widths, hist_values), ...]
    datasets = [
        (nc["diameter_bin_center"][:][4:20], 
        nc["diameter_bin_width"][:][4:20], 
        np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20])
        for i in range(100)
    ]

    initial_params = [1.9935, 0.735, 0.0365]
    fitted_c0, fitted_c1, fitted_c2 = optimize_meta_params(datasets, initial_params, lr=1e-3, max_iter=100)
    print(f"Optimized meta-parameters: c0={fitted_c0:.4f}, c1={fitted_c1:.4f}, c2={fitted_c2:.4f}")


In [ ]:
from torch.distributions import Gamma
import torch

def compute_negative_log_likelihood(mu, bin_centers, bin_widths, hist_values, c0, c1, c2):
    scale = c0+c1*mu+c2*mu*mu
    shape = mu + 1  # shape = alpha
    if scale <= 1e-6:
        return torch.tensor(float('inf'), dtype=torch.float64)  # Avoid invalid scale
    
    # Use PyTorch Gamma distribution
    gamma_dist = Gamma(shape, scale)  # PyTorch uses rate = 1/scale
    #print(gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp())
    #print(torch.from_numpy(np.array(bin_widths)))
    pdf_values = gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp() * torch.from_numpy(np.array(bin_widths))
    hist_values = torch.tensor(np.array(hist_values).astype(np.float64), dtype=torch.float64)
    print(pdf_values)

    # Negative log-likelihood
    return -torch.sum(hist_values * torch.log(pdf_values) - pdf_values)

datasets = [
    (np.array(nc["diameter_bin_center"][:][4:20]), 
    np.array(nc["diameter_bin_width"][:][4:20]), 
    np.array(np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20]))
    for i in range(1)
]
c0, c1, c2 = [1.9935, 0.735, 0.0365]


compute_negative_log_likelihood(
    mu=0.2, 
    bin_centers=np.array(nc["diameter_bin_center"][:][4:20]), 
    bin_widths=np.array(nc["diameter_bin_width"][:][4:20]), 
    hist_values=np.array(np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20]), 
    c0=c0, 
    c1=c1, 
    c2=c2
)

In [ ]:
import torch
from torch.distributions import Gamma

# Define the scale computation
def compute_scale(mu, c0, c1, c2):
    return c0 + c1 * mu + c2 * mu**2

# Compute the negative log-likelihood for a single histogram
def compute_negative_log_likelihood(mu, bin_centers, bin_widths, hist_values, c0, c1, c2):
    scale = compute_scale(mu, c0, c1, c2)
    shape = mu + 1  # shape = alpha
    if scale <= 1e-6:
        return torch.tensor(float('inf'), dtype=torch.float64)  # Avoid invalid scale
    
    # Use PyTorch Gamma distribution
    gamma_dist = Gamma(shape, scale)  # PyTorch uses rate = 1/scale
    #print(gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp())
    #print(torch.from_numpy(np.array(bin_widths)))
    pdf_values = gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp() * torch.from_numpy(np.array(bin_widths))
    hist_values = torch.tensor(np.array(hist_values).astype(np.float64), dtype=torch.float64)

    # Negative log-likelihood
    return -torch.sum(hist_values * torch.log(pdf_values) - pdf_values)

# Optimize mu for a single dataset using gradient descent
def optimize_mu_manual(bin_centers, bin_widths, hist_values, c0, c1, c2, lr=1e-2, max_iter=100, tol=1e-6):
    # Initialize mu
    mu = torch.tensor(2.0, dtype=torch.float64, requires_grad=True)  # Initial guess

    optimizer = torch.optim.SGD([mu], lr=lr)

    for iteration in range(max_iter):
        # Zero the gradients
        optimizer.zero_grad()

        # Compute the negative log-likelihood
        nll = compute_negative_log_likelihood(mu, bin_centers, bin_widths, hist_values, c0, c1, c2)

        # Backpropagate to compute gradients
        nll.backward()

        # Update mu
        optimizer.step()
        

        # Check for convergence
        if abs(mu.grad.item()) < tol:
            break

    return mu.item()

# Compute the total negative log-likelihood for all datasets
def compute_total_log_likelihood(datasets, c0, c1, c2, lr_mu=1e-2, max_iter_mu=100):
    total_ll = torch.tensor(0.0, requires_grad=True)
    for bin_centers, bin_widths, hist_values in datasets:
        # Optimize mu for the current histogram
        mu = optimize_mu_manual(bin_centers, bin_widths, hist_values, c0, c1, c2, lr=lr_mu, max_iter=max_iter_mu)
        scale = compute_scale(mu, c0, c1, c2)
        shape = mu + 1

        # Compute PDF using PyTorch Gamma distribution
        gamma_dist = Gamma(shape, 1.0 / scale)  # PyTorch uses rate = 1/scale
        pdf_values = gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp() * torch.from_numpy(np.array(bin_widths))
        hist_values = torch.tensor(np.array(hist_values).astype(np.float64), dtype=torch.float64)

        # Accumulate negative log-likelihood
        total_ll = total_ll - torch.sum(hist_values * torch.log(pdf_values) - pdf_values).item()

    return total_ll

# Gradient descent for c0, c1, c2 using PyTorch
def optimize_meta_params(datasets, initial_params, lr=1e-3, max_iter=100, tol=1e-6):
    # Initialize parameters as PyTorch tensors with gradients enabled
    c0 = torch.tensor(initial_params[0], dtype=torch.float64, requires_grad=True)
    c1 = torch.tensor(initial_params[1], dtype=torch.float64, requires_grad=True)
    c2 = torch.tensor(initial_params[2], dtype=torch.float64, requires_grad=True)

    optimizer = torch.optim.Adam([c0, c1, c2], lr=lr)

    for iteration in range(max_iter):
        # Zero the gradients
        optimizer.zero_grad()

        # Compute the total negative log-likelihood
        total_ll = compute_total_log_likelihood(datasets, c0, c1, c2)

        # Backpropagate to compute gradients
        total_ll.backward()

        # Update parameters
        optimizer.step()

        # Print progress
        print(f"Iteration {iteration}: c0={c0.item():.4f}, c1={c1.item():.4f}, c2={c2.item():.4f}, Total LL={total_ll:.4f}")

        # Check for convergence
        if abs(total_ll) < tol:
            print("Convergence achieved!")
            break

    return c0.item(), c1.item(), c2.item()

# Example usage
datasets = [
    (np.array(nc["diameter_bin_center"][:][4:20]), 
    np.array(nc["diameter_bin_width"][:][4:20]), 
    np.array(np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20]))
    for i in range(100)
]

initial_params = [1.9935, 0.735, 0.0365]
fitted_c0, fitted_c1, fitted_c2 = optimize_meta_params(datasets, initial_params, lr=3e-3, max_iter=1000)
print(f"Optimized meta-parameters: c0={fitted_c0:.4f}, c1={fitted_c1:.4f}, c2={fitted_c2:.4f}")


In [ ]:
gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float32)).exp()

In [ ]:
hist_values = np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20]
    #hist_values = np.random.poisson(10, size=len(bin_edges) - 1)

    # Compute bin centers and bin widths
    bin_centers = nc["diameter_bin_center"][:][4:20]
    bin_widths = nc["diameter_bin_width"][:][4:20]

In [ ]:
def total_negative_log_likelihood(meta_params, datasets):
    c0, c1, c2 = meta_params
    total_nll = 0.0

    for dataset in datasets:
        bin_centers, bin_widths, hist_values = dataset

        # Fit shape for this dataset
        try:
            fitted_shape = fit_shape_given_meta_params(c0, c1, c2, bin_centers, bin_widths, hist_values)
        except RuntimeError:
            return np.inf  # Skip invalid fits

        # Compute negative log-likelihood for this dataset
        mu = fitted_shape - 1
        scale = c0 + c1 * mu + c2 * mu**2
        pdf_values = gamma.pdf(bin_centers, a=fitted_shape, scale=scale) * bin_widths
        total_nll += -np.sum(hist_values * np.log(pdf_values) - pdf_values)

    return total_nll


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

X = np.array([np.array(fitted_shape1_list)-1, 1/np.array(fitted_scale1_list)]).T
X_pca = pca.fit(X)
print(pca.explained_variance_ratio_)
print(pca.singular_values_)

In [ ]:
plt.scatter(X_pca.transform(X)[:, 0], X_pca.transform(X)[:, 1]) 

In [ ]:
X = np.array([[-1, -1], [-2, -1], [-3, -2], [1, 1], [2, 1], [3, 2]])
X.shape

In [ ]:
y = np.sum(nc["raw_drop_number"][:], axis=2)[4][4:20]
def negative_log_likelihood_constrained_scalar(shape):
    #if shape <= 0:
    #    return np.inf
    mu = shape - 1
    scale = 1/(1.935 + 0.735 * mu + 0.0365 * mu**2)
    if scale <= 1e-6:  # Avoid scale being too small
        return np.inf
    pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
    pdf_values = np.maximum(pdf_values, 1e-100)
    return -np.sum(y * np.log(pdf_values) - pdf_values)
result = minimize_scalar(negative_log_likelihood_constrained_scalar)
result

In [ ]:
pdf =  gamma.pdf(x_values, a=5, scale=0.2)
plt.figure(figsize=(8, 5))
plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")
plt.plot(x_values, pdf, label=f"Fitted Gamma PDF\n(shape={fitted_shape1:.2f}, scale={fitted_scale1:.2f})", color="red")
#plt.plot(x_values, fitted_pdf2, label=f"Fitted Gamma PDF\n(shape={fitted_shape2:.2f}, scale={fitted_scale2:.2f})", color="blue")


In [ ]:
hist_values = np.sum(nc["raw_drop_number"][:], axis=2)[3][4:20]
hist_values_normalized = hist_values / (hist_values.sum() * bin_widths)
pdf = gamma.pdf(x_values, a=5, scale=0.2)
plt.figure(figsize=(8, 5))
plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")
#plt.plot(x_values, pdf, label=f"Fitted Gamma PDF\n(shape={fitted_shape1:.2f}, scale={fitted_scale1:.2f})", color="red")
for shape in range(4, 8, 1):
    mu = shape - 1
    scale = 1/(1.935 + 0.735 * mu + 0.0365 * mu**2)
    pdf =  gamma.pdf(x_values, a=shape, scale=scale)
    plt.plot(x_values, pdf, label=f"Fitted Gamma PDF\n(shape={shape:.2f}, scale={scale:.2f})")
    plt.xlim(0, 3)
    print(negative_log_likelihood_constrained_scalar(shape))
    
plt.legend()

In [ ]:
hist_values = np.sum(nc["raw_drop_number"][:], axis=2)[4][4:20]
for i in range(1, 20):
    print(i, negative_log_likelihood_constrained(i, bin_centers, hist_values, bin_widths))
#plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")

In [ ]:
X = [0, 2]
plt.figure(figsize=(6, 6))
plt.scatter(D1list, D2list)
plt.xlim(0, 2)
plt.ylim(0, 2)
plt.plot(X, X)
plt.xlabel("D0 with full Gamma")
plt.ylabel("D0 with constrained Gamma")
plt.savefig("")

In [ ]:
for i in range(1, 20):
    print(i, negative_log_likelihood_constrained(i, bin_centers, hist_values, bin_widths))

In [ ]:
ilist = np.linspace(11, 12, 101)
for i in ilist:
    print(i, negative_log_likelihood_constrained(i, bin_centers, hist_values, bin_widths))

In [ ]:
shape2 = 11.4
mu2 = shape2 - 1
scale2 = 1/(1.935 + 0.735 * mu2 + 0.0365 * mu2**2)
fitted_pdf2 = gamma.pdf(x_values, a=fitted_shape2, scale=fitted_scale2)
plt.figure(figsize=(8, 5))
plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")
plt.plot(x_values, fitted_pdf1, label=f"Fitted Gamma PDF\n(shape={fitted_shape1:.2f}, scale={fitted_scale1:.2f})", color="red")
plt.plot(x_values, fitted_pdf2, label=f"Fitted Gamma PDF\n(shape={fitted_shape2:.2f}, scale={fitted_scale2:.2f})", color="blue")
plt.annotate(f"{D1=}", [4, 0.7])
plt.annotate(f"{D2=}", [4, 0.5])
plt.xlim(0, 5)
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.title("Gamma Distribution Fit to Unequal Bin Histogram")
#plt.show()
plt.savefig(f"GammaFit")

In [ ]:
result2